# Práctica 1 - Visualización de datos

El archivo contiene mediciones diarias de temperatura máxima (`TMAX`) y mínima (`TMIN`) de estaciones cercanas a Ann Arbor. El valor está expresado en décimas de grado Celsius, por eso se divide entre 10 para obtener grados Celsius.

Para cada día del año voy a calcular los récords de temperatura máxima y mínima del período 2005-2014. Después voy a comparar esos valores con las mediciones de 2015 y mostrar los días en que se superó alguno de los récords.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# El notebook puede ejecutarse desde la carpeta notebooks o desde la raíz del repositorio.
csv_name = "fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89.csv"
path_from_notebooks = Path("../data") / csv_name
path_from_root = Path("data") / csv_name
data_path = path_from_notebooks if path_from_notebooks.exists() else path_from_root

df = pd.read_csv(data_path)
df.head()


## Preparación de los datos

Convierto la fecha a formato de fecha, paso las temperaturas a grados Celsius y creo una columna con el mes y el día. Quito el 29 de febrero para que cada serie tenga los mismos 365 días.


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
df["Month_Day"] = df["Date"].dt.strftime("%m-%d")
df["Temperature_C"] = df["Data_Value"] / 10

# El 29 de febrero no se incluye en esta comparación.
df = df[~((df["Date"].dt.month == 2) & (df["Date"].dt.day == 29))].copy()

df["Element"].value_counts()


## Récords entre 2005 y 2014

Agrupo por mes y día, sin importar el año. Para `TMAX` tomo el valor más alto y para `TMIN` el más bajo. En 2015 conservo solo las mediciones que son estrictamente mayores o menores que el récord correspondiente.


In [ ]:
historical = df[df["Year"].between(2005, 2014)]

record_high = (
    historical[historical["Element"] == "TMAX"]
    .groupby("Month_Day")["Temperature_C"]
    .max()
)
record_low = (
    historical[historical["Element"] == "TMIN"]
    .groupby("Month_Day")["Temperature_C"]
    .min()
)

data_2015 = df[df["Year"] == 2015]
tmax_2015 = data_2015[data_2015["Element"] == "TMAX"].copy()
tmin_2015 = data_2015[data_2015["Element"] == "TMIN"].copy()

# Se compara cada medición de 2015 con el récord del mismo mes y día.
tmax_2015 = tmax_2015[
    tmax_2015["Temperature_C"] > tmax_2015["Month_Day"].map(record_high)
]
tmin_2015 = tmin_2015[
    tmin_2015["Temperature_C"] < tmin_2015["Month_Day"].map(record_low)
]

# Fechas de referencia de un año no bisiesto para ordenar y graficar los días del año.
record_high.index = pd.to_datetime("2001-" + record_high.index)
record_low.index = pd.to_datetime("2001-" + record_low.index)

print(f"Días con récord máximo superado en 2015: {len(tmax_2015)}")
print(f"Días con récord mínimo superado en 2015: {len(tmin_2015)}")


## Visualización

Las líneas muestran los récords de 2005-2014 y el área sombreada representa el intervalo entre el récord mínimo y el máximo de cada día. Los puntos señalan las mediciones de 2015 que superaron esos valores.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(record_high.index, record_high.values, color="#c44e52", linewidth=1.3,
        label="Récord máximo 2005-2014")
ax.plot(record_low.index, record_low.values, color="#4c72b0", linewidth=1.3,
        label="Récord mínimo 2005-2014")
ax.fill_between(record_high.index, record_low.values, record_high.values,
                color="#9bbbd4", alpha=0.22, label="Rango entre récords")

ax.scatter(pd.to_datetime("2001-" + tmax_2015["Month_Day"]),
           tmax_2015["Temperature_C"], color="#e1812c", s=22, zorder=3,
           label="Máximo de 2015 que superó el récord")
ax.scatter(pd.to_datetime("2001-" + tmin_2015["Month_Day"]),
           tmin_2015["Temperature_C"], color="#8172b3", s=22, zorder=3,
           label="Mínimo de 2015 que superó el récord")

ax.set_title("Temperaturas récord diarias y récords superados en 2015", fontsize=14, pad=12)
ax.set_xlabel("Día del año")
ax.set_ylabel("Temperatura (°C)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.grid(axis="y", color="#d9d9d9", linewidth=0.7, alpha=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.14))
fig.tight_layout()
plt.show()
